# UAS Machine Learning — CRISP-DM
## Prediksi Sales menggunakan Random Forest Regressor

**Dataset:** Superstore Sales (subset 84 transaksi untuk demo/reproducibility)  
**Sumber publik:** Kaggle / GitHub Sample Superstore  
**Target:** `Sales`  
**Model:** Random Forest Regressor  
**Deployment:** Streamlit

Sumber:
- https://www.kaggle.com/datasets/himanshuuike/superstore-sales-dataset
- https://github.com/leonism/sample-superstore

> Dataset `data/superstore.csv` yang disertakan adalah subset transaksi dari dataset publik tersebut agar project dapat langsung dijalankan secara lokal/Streamlit tanpa mengunduh data lagi.


## 1. Business Understanding

### Masalah
Bisnis retail membutuhkan estimasi nilai penjualan berdasarkan karakteristik transaksi.

### Tujuan
Membangun model regresi yang memperkirakan nilai `Sales` menggunakan informasi transaksi seperti metode pengiriman, segmen pelanggan, wilayah, kategori, subkategori, jumlah barang, diskon, dan waktu transaksi.

### Manfaat
Prototype dapat digunakan untuk memahami pola penjualan dan memberikan estimasi nilai transaksi melalui dashboard.


## 2. Data Understanding

Atribut utama:
- `order_date`: tanggal transaksi
- `ship_mode`: metode pengiriman
- `segment`: segmen pelanggan
- `region`: wilayah
- `category`: kategori produk
- `sub_category`: subkategori produk
- `sales`: target nilai penjualan
- `quantity`: jumlah barang
- `discount`: diskon


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATA_PATH = "../data/superstore.csv"
MODEL_PATH = "../model/sales_model.pkl"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())


In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplikat:", df.duplicated().sum())
print("\nStatistik:")
display(df.describe())


## 3. Data Preparation

Tahapan:
1. Mengubah tanggal ke tipe datetime.
2. Menghapus duplikat.
3. Menangani nilai kosong melalui imputer pada pipeline.
4. Membuat fitur tahun, bulan, hari, dan weekday.
5. One-Hot Encoding untuk fitur kategorikal.
6. Membagi data 80:20 untuk training/testing.


In [ ]:
data = df.copy()
data["order_date"] = pd.to_datetime(data["order_date"], errors="coerce")
data = data.drop_duplicates().dropna(subset=["sales"]).reset_index(drop=True)

data["order_year"] = data["order_date"].dt.year
data["order_month"] = data["order_date"].dt.month
data["order_day"] = data["order_date"].dt.day
data["order_weekday"] = data["order_date"].dt.dayofweek

features = [
    "ship_mode","segment","region","category","sub_category",
    "quantity","discount","order_year","order_month","order_day","order_weekday"
]

X = data[features]
y = data["sales"]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)


In [ ]:
# EDA
category_sales = data.groupby("category")["sales"].sum().sort_values(ascending=False)
display(category_sales)

fig, ax = plt.subplots(figsize=(8,4))
category_sales.plot(kind="bar", ax=ax)
ax.set_title("Total Sales per Category")
ax.set_ylabel("Sales")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. Modeling — Random Forest Regressor

Random Forest dipilih karena target `Sales` berbentuk numerik kontinu. Preprocessing dan model dibuat dalam satu Pipeline sehingga proses yang sama digunakan saat training dan prediksi.


In [ ]:
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
print("Training selesai.")


## 5. Evaluation

Metrik:
- MAE
- RMSE
- R²

Karena dataset demo hanya merupakan subset kecil, hasil metrik digunakan sebagai hasil pembelajaran/prototype dan bukan sebagai klaim performa produksi.


In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:.4f}")

comparison = pd.DataFrame({"Actual": y_test.values, "Predicted": y_pred})
display(comparison.head(10))

fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(y_test, y_pred, alpha=.7)
ax.set_xlabel("Actual Sales")
ax.set_ylabel("Predicted Sales")
ax.set_title("Actual vs Predicted")
plt.tight_layout()
plt.show()


## 6. Deployment

Model disimpan ke `model/sales_model.pkl`. Aplikasi Streamlit memuat file ini sehingga **tidak melakukan training ulang saat aplikasi dibuka**.


In [ ]:
with open(MODEL_PATH, "wb") as f:
    pickle.dump(model, f)

print("Model tersimpan di:", MODEL_PATH)

# Uji load kembali
with open(MODEL_PATH, "rb") as f:
    loaded_model = pickle.load(f)

sample_prediction = loaded_model.predict(X_test.iloc[[0]])[0]
print("Contoh prediksi setelah load ulang:", round(sample_prediction, 2))


## 7. Kesimpulan

Project telah menerapkan seluruh tahapan CRISP-DM:
1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment

Model Random Forest diekspor ke Pickle dan digunakan oleh frontend Streamlit.
